In [1]:
# Importing needed Libraries
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MultiLabelBinarizer

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.multioutput import MultiOutputClassifier

In [2]:
# Reading the processed Data
df = pd.read_excel("../Processed_data.xlsx")
df.head(100)

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,46,Male,95_left.jpg,suspected glaucoma,0,0,1,0,0,0,0,0
96,96,56,Female,96_left.jpg,vitreous degeneration,0,0,0,0,0,0,0,1
97,97,64,Female,97_left.jpg,normal fundus,1,0,0,0,0,0,0,0
98,98,69,Male,98_left.jpg,branch retinal vein occlusion,0,0,0,0,0,0,0,1


# Preprocessing

In this section we are going to first create a target variable out of the various labels for our multi-label classification problem. Next will be to Encode the data accordingly. Lastly, split the data into training and testing, we will then apply the other preprocessing steps to each split seperately. 

The objectives are to bin and encode the age variable, and the diagnosis columns.

We are not going to need the ID column for the classification task, hence we will drop that column. Other stages of work to be done, are as follows:
- Encode the Patient Sex column, so it is numerically 0 and 1. 
- One Hot Encode the Diagnosis Columns. The vector will be large and maybe a sparse matrix would be better equipped, but it should be interesting to see how many layers the decision tree will need with this information available. 

In [3]:
# Creating the target Labels 
def createTarget(df):
    df['Labels'] = [[0,0,0,0,0,0,0] for i in range (0,len(df))] #create a column for labels with 8 0s
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
    label = [] #create empty label array
    for i in range(0, len(df)):
        for target in target_columns:
            label.append(df.loc[i, target]) #append the value for the column for each row to the label
        
        df.at[i,'Labels'] = label #update the label column with the label array
        label = [] #reset label array
            

In [4]:
createTarget(df)

In [5]:
df.head()

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O,Labels
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1,"[0, 1, 0, 0, 0, 0, 0, 1]"
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"


We're going to drop the Patient ID columns and the previous label columns to create the singular array in a copy of the df array.

In [6]:
#Dropping ID column and original label columns
res_df = df.drop(['Patient ID', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O'], axis=1)

In [7]:
res_df.head()

,Patient Age,Patient Sex,Filename,Diagnosis,Labels
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]"
3,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"


In [8]:
# one hot encoder object to be used for binned age group
one_hot = OneHotEncoder()

# label encoder to be used for sex
label_encoder = LabelEncoder() 

# multi label binarizer object to be used for diagnosis
mlb = MultiLabelBinarizer() 

In [9]:
# Encoding the Patient Sex
res_df['Patient Sex'] = label_encoder.fit_transform(res_df['Patient Sex']) 
res_df.head()

,Patient Age,Patient Sex,Filename,Diagnosis,Labels
0,69,0,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,1,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,42,1,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]"
3,66,1,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,53,1,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"


In [10]:
# bins for age
bins = [0, 20,30,40,50,60,70,80,100]

# labels for the age bins
labels = ['<=20','21-30','31-40','41-50','51-60','61-70','71-80','80+']

In [11]:
# binning the age group and creating the column Age group
res_df['Age Group'] = pd.cut(res_df['Patient Age'], bins = bins, labels = labels, right=True)

# drop patient age column
res_df.drop("Patient Age", axis=1, inplace=True) 

res_df.head()

,Patient Sex,Filename,Diagnosis,Labels,Age Group
0,0,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]",61-70
1,1,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",51-60
2,1,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]",41-50
3,1,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",61-70
4,1,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]",51-60


In [12]:
#one hot encoding diagnosis columns
res_df['Diagnosis'] = res_df['Diagnosis'].str.split(',')

# Apply MultiLabelBinarizer
_df = pd.DataFrame(mlb.fit_transform(res_df['Diagnosis']), columns=mlb.classes_, index=df.index)
res_df = pd.concat([res_df, _df], axis=1)
res_df.drop('Diagnosis',axis=1,inplace=True)

res_df. head(10)

,Patient Sex,Filename,Labels,Age Group,abnormal pigment,age-related macular degeneration,anterior segment image,arteriosclerosis,asteroid hyalosis,atrophic change,...,suspicious diabetic retinopathy,tessellated fundus,vascular loops,vessel tortuosity,vitreous degeneration,vitreous opacity,wedge white line change,wedge-shaped change,wet age-related macular degeneration,white vessel
0,0,0_left.jpg,"[0, 0, 0, 1, 0, 0, 0, 0]",61-70,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",51-60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,2_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 1]",41-50,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,3_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",61-70,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,4_left.jpg,"[0, 0, 0, 0, 0, 0, 0, 1]",51-60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,5_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",41-50,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1,6_left.jpg,"[0, 0, 0, 0, 0, 0, 0, 1]",51-60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,7_left.jpg,"[0, 0, 0, 0, 0, 0, 0, 1]",51-60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,1,8_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",51-60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1,9_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",51-60,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
column_transformer = ColumnTransformer(
    transformers=[
        ('One Hot ', one_hot, ['Age Group'])
    ],remainder='passthrough')

In [14]:
res_df = pd.DataFrame(column_transformer.fit_transform(res_df), columns=column_transformer.get_feature_names_out(res_df.columns))
res_df.head(10)


,One Hot __Age Group_21-30,One Hot __Age Group_31-40,One Hot __Age Group_41-50,One Hot __Age Group_51-60,One Hot __Age Group_61-70,One Hot __Age Group_71-80,One Hot __Age Group_80+,One Hot __Age Group_<=20,remainder__Patient Sex,remainder__Filename,...,remainder__suspicious diabetic retinopathy,remainder__tessellated fundus,remainder__vascular loops,remainder__vessel tortuosity,remainder__vitreous degeneration,remainder__vitreous opacity,remainder__wedge white line change,remainder__wedge-shaped change,remainder__wet age-related macular degeneration,remainder__white vessel
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0_left.jpg,...,0,0,0,0,0,0,0,0,0,0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,1_left.jpg,...,0,0,0,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,2_left.jpg,...,0,0,0,0,0,0,0,0,0,0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1,3_left.jpg,...,0,0,0,0,0,0,0,0,0,0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,4_left.jpg,...,0,0,0,0,0,0,0,0,0,0
5,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,5_left.jpg,...,0,0,0,0,0,0,0,0,0,0
6,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,6_left.jpg,...,0,0,0,0,0,0,0,0,0,0
7,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,7_left.jpg,...,0,0,0,0,0,0,0,0,0,0
8,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,8_left.jpg,...,0,0,0,0,0,0,0,0,0,0
9,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,9_left.jpg,...,0,0,0,0,0,0,0,0,0,0


In [15]:
for col in res_df:
    if col[0:10] == "One Hot __":
        res_df.rename(columns={col:col[10:]},inplace=True)
    elif col[0:11] == "remainder__":
        res_df.rename(columns={col:col[11:]},inplace=True)
    
res_df.head(20)

,Age Group_21-30,Age Group_31-40,Age Group_41-50,Age Group_51-60,Age Group_61-70,Age Group_71-80,Age Group_80+,Age Group_<=20,Patient Sex,Filename,...,suspicious diabetic retinopathy,tessellated fundus,vascular loops,vessel tortuosity,vitreous degeneration,vitreous opacity,wedge white line change,wedge-shaped change,wet age-related macular degeneration,white vessel
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0_left.jpg,...,0,0,0,0,0,0,0,0,0,0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,1_left.jpg,...,0,0,0,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,2_left.jpg,...,0,0,0,0,0,0,0,0,0,0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1,3_left.jpg,...,0,0,0,0,0,0,0,0,0,0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,4_left.jpg,...,0,0,0,0,0,0,0,0,0,0
5,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,5_left.jpg,...,0,0,0,0,0,0,0,0,0,0
6,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,6_left.jpg,...,0,0,0,0,0,0,0,0,0,0
7,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,7_left.jpg,...,0,0,0,0,0,0,0,0,0,0
8,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,8_left.jpg,...,0,0,0,0,0,0,0,0,0,0
9,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1,9_left.jpg,...,0,0,0,0,0,0,0,0,0,0


In [16]:
y = res_df['Labels'].apply(lambda x: np.array(x))
y = np.array(y.tolist()) 
X = res_df.drop(['Filename','Labels'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
classifier = DecisionTreeClassifier(random_state=42)
multi_target_decision = MultiOutputClassifier(classifier, n_jobs=2)

# Train the classifier on the training data
multi_target_decision.fit(X_train, y_train)

# Make predictions on the test data
y_pred = multi_target_decision.predict(X_test)

# Evaluate the performance
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       631
           1       1.00      1.00      1.00       359
           2       1.00      1.00      1.00        74
           3       1.00      1.00      1.00        75
           4       1.00      1.00      1.00        54
           5       1.00      1.00      1.00        43
           6       1.00      1.00      1.00        53
           7       1.00      0.99      1.00       262

   micro avg       1.00      1.00      1.00      1551
   macro avg       1.00      1.00      1.00      1551
weighted avg       1.00      1.00      1.00      1551
 samples avg       1.00      1.00      1.00      1551



C:\Users\mimiw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [18]:
# The predicted outcome
y_pred

array([[0, 1, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1]], dtype=int64)

In [19]:
# Checking the predicted outcome with the actual outcome
y_test == y_pred

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]])

In [20]:
y = res_df['Labels'].apply(lambda x: np.array(x))
y = np.array(y.tolist()) 
X = res_df.drop(['Filename','Labels','Age Group_21-30','Age Group_31-40','Age Group_41-50','Age Group_51-60','Age Group_61-70','Age Group_71-80','Age Group_80+','Age Group_<=20'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
classifier = DecisionTreeClassifier(random_state=42)
multi_target_decision = MultiOutputClassifier(classifier, n_jobs=2)
# Train the classifier on the training data
multi_target_decision.fit(X_train, y_train)

# Make predictions on the test data
y_pred = multi_target_decision.predict(X_test)

# Evaluate the performance
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       631
           1       1.00      1.00      1.00       359
           2       1.00      1.00      1.00        74
           3       1.00      1.00      1.00        75
           4       1.00      1.00      1.00        54
           5       1.00      1.00      1.00        43
           6       1.00      1.00      1.00        53
           7       1.00      0.99      1.00       262

   micro avg       1.00      1.00      1.00      1551
   macro avg       1.00      1.00      1.00      1551
weighted avg       1.00      1.00      1.00      1551
 samples avg       1.00      1.00      1.00      1551



C:\Users\mimiw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
